# 02 — Modelo final de demanda de taxis

Este notebook parte del dataset limpio y agregado que se genera en `022_modelo_batch.ipynb` y queda guardado en `../data_ml/df_ml`.

Aqui no se repite la limpieza ni la creacion de variables. El objetivo es cargar `df_ml`, aplicar los mejores hiperparametros obtenidos por validacion cruzada, comparar las familias de modelos en validation, reentrenar el mejor modelo con `train + validation` y guardarlo para streaming.

## Carga de `df_ml`

`df_ml` ya contiene la demanda agregada cada 15 minutos por zona, las variables temporales y los lags exactos usados por streaming: `lag_1` y `lag_2`.

In [ ]:
from pathlib import Path
import json
import time

import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import (
    LinearRegression,
    GeneralizedLinearRegression,
    DecisionTreeRegressor,
    RandomForestRegressor,
    GBTRegressor,
)
from pyspark.ml.evaluation import RegressionEvaluator

try:
    spark.stop()
except NameError:
    pass

spark = (
    SparkSession.builder
    .appName("nyc-taxi-final-model")
    .master("local[*]")
    .config("spark.executor.memory", "4g")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

df_ml = spark.read.parquet("../data_ml/df_ml")

print(f"df_ml cargado: {df_ml.count():,} filas")
df_ml.printSchema()

## 2.2 MODELOS ML

En esta libreta se parte de los mejores hiperparametros obtenidos en `022_modelo_batch.ipynb`. Cada familia se entrena con `train`, se compara en `validation` y el mejor modelo global se reentrena con `train + validation` antes de evaluarlo una sola vez sobre `test`.

### Preparacion del dataset para ML

Se usan exactamente las mismas variables que en la validacion cruzada y que en el consumidor de streaming.

In [ ]:
feature_cols = [
    "zone_lon",
    "zone_lat",
    "hour",
    "dayofweek",
    "is_weekend",
    "is_rush_hour",
    "is_late_night",
    "lag_1",
    "lag_2",
]

label_col = "label"

missing_cols = [c for c in feature_cols + ["trip_count", "window_start"] if c not in df_ml.columns]
if missing_cols:
    raise ValueError(f"Faltan columnas en df_ml: {missing_cols}")

df_model = (
    df_ml
    .select(*feature_cols, "trip_count", "window_start")
    .dropna()
    .withColumn(label_col, F.col("trip_count").cast("double"))
)

train_df = df_model.filter(F.col("window_start") < F.lit("2009-01-22 00:00:00").cast("timestamp"))
val_df = df_model.filter(
    (F.col("window_start") >= F.lit("2009-01-22 00:00:00").cast("timestamp"))
    & (F.col("window_start") < F.lit("2009-01-27 00:00:00").cast("timestamp"))
)
test_df = df_model.filter(F.col("window_start") >= F.lit("2009-01-27 00:00:00").cast("timestamp"))
train_val_df = train_df.unionByName(val_df)

for split_df in [train_df, val_df, test_df, train_val_df]:
    split_df.cache()

split_counts = pd.DataFrame([
    {"split": "train", "filas": train_df.count()},
    {"split": "validation", "filas": val_df.count()},
    {"split": "test", "filas": test_df.count()},
    {"split": "train+validation", "filas": train_val_df.count()},
])

display(split_counts)

### Entrenamiento y comparacion de familias

Se evalua cada modelo con sus mejores parametros sobre `train` y `validation`. La seleccion global se hace por MAE de validacion, porque es la metrica mas interpretable para la demanda: error medio en numero de viajes.

In [ ]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

fallback_best_params = {
    "Regresion lineal": {"regParam": 0.1, "elasticNetParam": 0.0},
    "Regresion Poisson": {"regParam": 0.1},
    "Decision Tree": {"maxDepth": 12, "minInstancesPerNode": 1},
    "Random Forest": {"numTrees": 80, "maxDepth": 12, "minInstancesPerNode": 1},
    "Gradient-Boosted Trees": {"maxIter": 120, "maxDepth": 5, "stepSize": 0.1},
}

params_path = Path("../models/cv_best_params.json")
if params_path.exists():
    with open(params_path, "r", encoding="utf-8") as f:
        best_params_by_model = json.load(f)
else:
    best_params_by_model = fallback_best_params


def build_estimator(model_name, params):
    if model_name == "Regresion lineal":
        return LinearRegression(featuresCol="features", labelCol=label_col, maxIter=50, **params)
    if model_name == "Regresion Poisson":
        return GeneralizedLinearRegression(
            featuresCol="features",
            labelCol=label_col,
            family="poisson",
            link="log",
            maxIter=50,
            **params,
        )
    if model_name == "Decision Tree":
        return DecisionTreeRegressor(featuresCol="features", labelCol=label_col, seed=42, **params)
    if model_name == "Random Forest":
        return RandomForestRegressor(featuresCol="features", labelCol=label_col, seed=42, **params)
    if model_name == "Gradient-Boosted Trees":
        return GBTRegressor(featuresCol="features", labelCol=label_col, seed=42, **params)
    raise ValueError(f"Modelo no soportado: {model_name}")


def evaluate_predictions(model_name, split_name, predictions, params, train_time_seconds):
    evaluators = {
        "RMSE": RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="rmse"),
        "MAE": RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="mae"),
        "R2": RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="r2"),
    }
    row = {
        "modelo": model_name,
        "split": split_name,
        "training_time_seconds": train_time_seconds,
        "params": params,
    }
    for metric_name, evaluator in evaluators.items():
        row[metric_name] = evaluator.evaluate(predictions)
    return row


trained_models = {}
comparison_rows = []

for model_name, params in best_params_by_model.items():
    estimator = build_estimator(model_name, params)
    pipeline = Pipeline(stages=[assembler, estimator])

    start = time.perf_counter()
    model = pipeline.fit(train_df)
    elapsed = time.perf_counter() - start
    trained_models[model_name] = model

    comparison_rows.append(evaluate_predictions(model_name, "train", model.transform(train_df), params, elapsed))
    comparison_rows.append(evaluate_predictions(model_name, "validation", model.transform(val_df), params, elapsed))

comparison_pdf = pd.DataFrame(comparison_rows).sort_values(["split", "MAE"])
validation_comparison_pdf = comparison_pdf[comparison_pdf["split"] == "validation"].sort_values("MAE")

best_model_name = validation_comparison_pdf.iloc[0]["modelo"]
best_model_params = best_params_by_model[best_model_name]

display(comparison_pdf)
print(f"Mejor modelo global segun MAE en validation: {best_model_name}")
print(f"Parametros: {best_model_params}")

### Reentrenamiento final y evaluacion en test

Una vez elegido el modelo global, se reentrena con `train + validation`. El conjunto `test` queda reservado para medir el resultado final sin haber participado en la seleccion.

In [ ]:
final_estimator = build_estimator(best_model_name, best_model_params)
final_pipeline = Pipeline(stages=[assembler, final_estimator])

start = time.perf_counter()
final_model = final_pipeline.fit(train_val_df)
final_train_time = time.perf_counter() - start

test_predictions = final_model.transform(test_df)
test_result = evaluate_predictions(
    best_model_name,
    "test",
    test_predictions,
    best_model_params,
    final_train_time,
)

test_results_pdf = pd.DataFrame([test_result])
display(test_results_pdf)

test_predictions.select(
    "window_start",
    "zone_lon",
    "zone_lat",
    "trip_count",
    "prediction",
).orderBy(F.col("prediction").desc()).show(20, truncate=False)

### Guardado del modelo final

El modelo final se guarda como `PipelineModel` para poder cargarlo directamente desde Spark Streaming.

In [ ]:
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

final_model_path = models_dir / "best_demand_model"
final_model.write().overwrite().save(str(final_model_path))

metadata = {
    "best_model_name": best_model_name,
    "best_params": best_model_params,
    "feature_cols": feature_cols,
    "label_col": label_col,
    "train_until": "2009-01-22 00:00:00",
    "validation_from": "2009-01-22 00:00:00",
    "validation_until": "2009-01-27 00:00:00",
    "test_from": "2009-01-27 00:00:00",
    "test_metrics": test_result,
}

with open(models_dir / "best_demand_model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)


print(f"Modelo final guardado en {final_model_path}")

### Importancia de variables

Si el modelo final es de arboles, se muestra la importancia de cada variable. En modelos lineales o Poisson esta grafica no aplica del mismo modo.

In [ ]:
last_stage = final_model.stages[-1]

if hasattr(last_stage, "featureImportances"):
    importances = list(last_stage.featureImportances)
    importance_pdf = (
        pd.DataFrame({"feature": feature_cols, "importance": importances})
        .sort_values("importance", ascending=False)
    )

    display(importance_pdf)

    ax = importance_pdf.plot.barh(x="feature", y="importance", figsize=(8, 5), legend=False)
    ax.invert_yaxis()
    ax.set_xlabel("Importancia")
    ax.set_ylabel("Variable")
    ax.set_title("Importancia de variables del modelo final")
    plt.tight_layout()
    plt.savefig("fig_feature_importance.png", dpi=150)
    plt.show()
else:
    print(f"El modelo {best_model_name} no expone featureImportances.")